In [ ]:
!pip install -U langchain langchain-community langchain-groq groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9
  Attempting uninstall

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b
@tool
def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="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"
)
llm_with_tools = llm.bind_tools([multiply, add, subtract])
tools_map = {
    "multiply": multiply,
    "add": add,
    "subtract": subtract
}


def run_agent_with_steps(query: str, max_iterations: int = 5):

    messages = [HumanMessage(content=query)]

    print("=" * 60)
    print("USER QUERY:", query)
    print("=" * 60, "\n")

    for i in range(max_iterations):

        print(f"--- Step {i+1}: LLM Thinking ---")

        response = llm_with_tools.invoke(messages)

        if response.tool_calls:

            print(f"Decision: Call {len(response.tool_calls)} tool(s)\n")
            messages.append(response)

            for tool_call in response.tool_calls:

                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                print(" TOOL CALL:", tool_name)
                print("   Arguments:", tool_args)

                tool_fn = tools_map[tool_name]
                result = tool_fn.invoke(tool_args)

                print("   Result:", result, "\n")

                messages.append(
                    ToolMessage(
                        content=str(result),
                        tool_call_id=tool_call["id"]
                    )
                )

        else:
            print("Decision: Provide final answer\n")
            print("=" * 60)
            print("FINAL ANSWER:", response.content)
            print("=" * 60)
            return response.content

    return "Max iterations reached"


run_agent_with_steps("What is 6 times 7?")


USER QUERY: What is 6 times 7?

--- Step 1: LLM Thinking ---
Decision: Call 1 tool(s)

🔧 TOOL CALL: multiply
   Arguments: {'a': 6, 'b': 7}
   Result: 42 

--- Step 2: LLM Thinking ---
Decision: Provide final answer

FINAL ANSWER: So, 6 times 7 is 42.


'So, 6 times 7 is 42.'